# ClinOntoScreen: NHANES + medical ontology + ML

This notebook downloads public NHANES XPT files, harmonizes the historical and later cycles, constructs a
leakage-safe cohort, loads the OWL/RDF screening ontology, trains Logistic Regression and XGBoost models,
evaluates ontology-assisted screening, and exports artifacts for the Streamlit dashboard and the paper.

**Primary question (H2, confirmatory):** at the same screening capacity, does an ontology-assisted ranking
identify more participants with HbA1c-defined screen-positive dysglycemia than XGBoost alone?

**Primary outcome:** among adults (18+) reporting neither prior diabetes nor prior prediabetes,
`LBXGH >= 5.7%`. This is a one-measurement research label, not a confirmed clinical diagnosis.

**Exploratory experiments (plan addendum §3):** E1 low-data learning curve, E2 missing-input stress test,
E3 model-family effect. They are reported whatever direction they take. H1, H3, H4 and all subgroup analyses
are exploratory; H2 is the only confirmatory test.

**Data periods:** NHANES 2017–March 2020 for training/validation; NHANES August 2021–August 2023 is kept as the
locked temporal test. CDC notes the later cycle has an updated sample design and modified
questionnaire/examination procedures, so results are temporal generalization, not evidence of a specific cause
of shift.

The notebook runs on CPU in Kaggle or Colab. Enable internet for data downloads. It does not use the temporal
test set for model selection. No performance result is assumed in advance.

In [ ]:
%pip -q install pandas numpy requests scikit-learn xgboost rdflib joblib matplotlib

from __future__ import annotations
from io import BytesIO
from pathlib import Path
from datetime import datetime, timezone
import json, os, platform, warnings, zipfile, subprocess, sys
from IPython.display import display
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from rdflib import Graph
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import joblib

# Paths are selected for Kaggle, Colab, or a local Jupyter runtime.
if os.environ.get('CLINONTOSCREEN_WORK_DIR'):
    WORK_DIR = Path(os.environ['CLINONTOSCREEN_WORK_DIR'])
elif Path('/kaggle/working').exists() and ('KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/input').exists()):
    WORK_DIR = Path('/kaggle/working/clinontoscreen_run')
elif 'google.colab' in sys.modules:
    WORK_DIR = Path('/content/clinontoscreen_run')
else:
    WORK_DIR = Path.cwd() / 'clinontoscreen_run'
CACHE_DIR = WORK_DIR / 'data_cache'
ARTIFACT_DIR = WORK_DIR / 'artifacts'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [17, 42, 123, 2025, 2026]
PRIMARY_CAPACITY = 0.20
CAPACITIES = [0.10, 0.20, 0.30]
N_BOOTSTRAP = 1000
LOW_DATA_FRACTIONS = [0.05, 0.10, 0.25, 0.50, 1.0]
LOW_DATA_REPEATS = 3
MASK_RATES = [0.0, 0.10, 0.30]
MASK_REPEATS = 3
MASK_COLUMNS = ['waist_cm', 'systolic_bp_mean', 'diastolic_bp_mean', 'hypertension_history',
                'vigorous_work_activity', 'moderate_work_activity',
                'vigorous_recreation_activity', 'moderate_recreation_activity']

def git_info():
    try:
        commit = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, timeout=10)
        tag = subprocess.run(['git', 'describe', '--tags', '--exact-match'], capture_output=True, text=True, timeout=10)
        return {
            'commit_hash': commit.stdout.strip() if commit.returncode == 0 else 'unavailable',
            'protocol_tag': tag.stdout.strip() if tag.returncode == 0 else 'not_tagged',
        }
    except Exception:
        return {'commit_hash': 'unavailable', 'protocol_tag': 'not_tagged'}

PROTOCOL = git_info()
print('Work directory:', WORK_DIR)
print('Python:', platform.python_version(), '| pandas:', pd.__version__, '| scikit-learn:', __import__('sklearn').__version__)
print('Protocol commit:', PROTOCOL['commit_hash'], '| tag:', PROTOCOL['protocol_tag'])
print('Artifacts will be written to:', ARTIFACT_DIR)

In [ ]:
# Official CDC public-use files. Optional files are logged if not available in a cycle.
CYCLE_CONFIGS = {
    '2017_mar2020': {'label': '2017–March 2020 pre-pandemic', 'base_url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/', 'prefix': 'P_', 'suffix': ''},
    '2021_2023': {'label': 'August 2021–August 2023', 'base_url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/', 'prefix': '', 'suffix': '_L'},
}
FILE_STEMS = ('DEMO', 'DIQ', 'GHB', 'BMX', 'BPXO', 'BPQ', 'SMQ', 'PAQ', 'MCQ', 'HDL')
ESSENTIAL_STEMS = {'DEMO', 'DIQ', 'GHB', 'BMX'}
FILE_VARIABLES = {
    'DEMO': ['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','DMDEDUC2','INDFMPIR','WTMECPRP','WTMEC2YR','SDMVSTRA','SDMVPSU'],
    'DIQ': ['SEQN','DIQ010','DIQ160'],
    'GHB': ['SEQN','LBXGH'],
    'BMX': ['SEQN','BMXBMI','BMXWAIST'],
    'BPXO': ['SEQN','BPXOSY1','BPXOSY2','BPXOSY3','BPXOSY4','BPXODI1','BPXODI2','BPXODI3','BPXODI4'],
    'BPQ': ['SEQN','BPQ020','BPQ030','BPQ040A'],
    'SMQ': ['SEQN','SMQ020'],
    'PAQ': ['SEQN','PAQ650','PAQ665','PAD660','PAD675','PAD680','PAD800','PAD820'],
    'MCQ': ['SEQN','MCQ160B','MCQ160C','MCQ160D','MCQ160E','MCQ160F','MCQ300C'],
    'HDL': ['SEQN','LBDHDD'],
}

def read_xpt(url: str, cache_path: Path, timeout: int = 90) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_sas(cache_path, format='xport', encoding='utf-8')
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(response.content)
    return pd.read_sas(BytesIO(response.content), format='xport', encoding='utf-8')

def load_cycle(cycle: str):
    cfg = CYCLE_CONFIGS[cycle]
    merged, report = None, {'cycle': cycle, 'label': cfg['label'], 'loaded_files': [], 'skipped_files': [], 'missing_variables': {}}
    for stem in FILE_STEMS:
        filename = f"{cfg['prefix']}{stem}{cfg['suffix']}.xpt"
        url = cfg['base_url'] + filename
        try:
            data = read_xpt(url, CACHE_DIR / cycle / filename)
        except Exception as exc:
            if stem in ESSENTIAL_STEMS:
                raise RuntimeError(f'Could not load essential CDC file {url}: {exc}') from exc
            report['skipped_files'].append({'file': filename, 'reason': str(exc)})
            print(f'Optional source skipped: {filename} ({exc})')
            continue
        wanted = FILE_VARIABLES[stem]
        report['missing_variables'][stem] = [c for c in wanted if c not in data.columns]
        data = data[[c for c in wanted if c in data.columns]].copy()
        if 'SEQN' not in data.columns:
            if stem in ESSENTIAL_STEMS:
                raise RuntimeError(f'Essential file {filename} does not contain SEQN')
            report['skipped_files'].append({'file': filename, 'reason': 'SEQN missing'})
            continue
        if merged is None:
            merged = data
        else:
            duplicate = (set(merged.columns) & set(data.columns)) - {'SEQN'}
            if duplicate:
                data = data.drop(columns=list(duplicate))
            merged = merged.merge(data, on='SEQN', how='left', validate='one_to_one')
        report['loaded_files'].append(filename)
    if merged is None:
        raise RuntimeError(f'No data loaded for cycle {cycle}')
    merged['cycle'] = cycle
    report['rows_after_merge'] = int(len(merged))
    return merged, report

def numeric(frame, col):
    if col not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[col], errors='coerce')

def clean_response(series, invalid=None):
    x = pd.to_numeric(series, errors='coerce').astype('float64')
    for bad in (invalid if invalid is not None else {7,9,77,99,777,999}):
        x = x.mask(x == bad)
    return x

def mean_available(frame, cols):
    usable = [c for c in cols if c in frame.columns]
    if not usable:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return frame[usable].apply(pd.to_numeric, errors='coerce').mean(axis=1, skipna=True)

def canonicalize(raw, cycle):
    out = pd.DataFrame(index=raw.index)
    out['SEQN'] = raw['SEQN']
    out['cycle'] = cycle
    out['age_years'] = numeric(raw, 'RIDAGEYR')
    out['sex_code'] = clean_response(raw['RIAGENDR'] if 'RIAGENDR' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['race_ethnicity_code'] = clean_response(raw['RIDRETH3'] if 'RIDRETH3' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['education_code'] = clean_response(raw['DMDEDUC2'] if 'DMDEDUC2' in raw else pd.Series(np.nan,index=raw.index))
    out['income_poverty_ratio'] = numeric(raw, 'INDFMPIR')
    out['bmi'] = numeric(raw, 'BMXBMI')
    out['waist_cm'] = numeric(raw, 'BMXWAIST')
    out['hdl_chol_mgdl'] = numeric(raw, 'LBDHDD')
    out['systolic_bp_mean'] = mean_available(raw, [f'BPXOSY{i}' for i in range(1,5)])
    out['diastolic_bp_mean'] = mean_available(raw, [f'BPXODI{i}' for i in range(1,5)])
    mapping = {
        'BPQ020':'hypertension_history','BPQ040A':'hypertension_treatment','SMQ020':'ever_smoked_100_cigs',
        'MCQ160B':'heart_failure_history','MCQ160C':'coronary_heart_disease_history','MCQ160D':'angina_history',
        'MCQ160E':'heart_attack_history','MCQ160F':'stroke_history'
    }
    for source, target in mapping.items():
        out[target] = clean_response(raw[source] if source in raw else pd.Series(np.nan,index=raw.index))
    # Harmonized leisure-time activity minutes. The two cycles use different instruments, so the
    # yes/no recreation items are unavailable in the later cycle; minutes are harmonized instead.
    special_minutes = {7777,9999,77777,99999}
    if cycle == '2017_mar2020':
        vigorous = clean_response(raw['PAD660'] if 'PAD660' in raw else pd.Series(np.nan,index=raw.index), special_minutes)
        vigorous = vigorous.where(clean_response(raw['PAQ650'] if 'PAQ650' in raw else pd.Series(np.nan,index=raw.index), invalid={7,9}) != 2, 0.0)
        moderate = clean_response(raw['PAD675'] if 'PAD675' in raw else pd.Series(np.nan,index=raw.index), special_minutes)
        moderate = moderate.where(clean_response(raw['PAQ665'] if 'PAQ665' in raw else pd.Series(np.nan,index=raw.index), invalid={7,9}) != 2, 0.0)
    else:
        vigorous = clean_response(raw['PAD820'] if 'PAD820' in raw else pd.Series(np.nan,index=raw.index), special_minutes).fillna(0.0)
        moderate = clean_response(raw['PAD800'] if 'PAD800' in raw else pd.Series(np.nan,index=raw.index), special_minutes).fillna(0.0)
    out['leisure_activity_minutes'] = (vigorous.fillna(0.0) + moderate.fillna(0.0)).where(vigorous.notna() | moderate.notna())
    out['sedentary_minutes'] = clean_response(raw['PAD680'] if 'PAD680' in raw else pd.Series(np.nan,index=raw.index), special_minutes)
    out['reported_diabetes'] = clean_response(raw['DIQ010'] if 'DIQ010' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['reported_prediabetes'] = clean_response(raw['DIQ160'] if 'DIQ160' in raw else pd.Series(np.nan,index=raw.index), invalid=set())
    out['hba1c_percent'] = numeric(raw, 'LBXGH')
    for weight_col in ('WTMECPRP','WTMEC2YR','SDMVSTRA','SDMVPSU'):
        if weight_col in raw.columns:
            out[weight_col.lower()] = numeric(raw, weight_col)
    return out

def make_cohort(frame):
    flow = {'rows_in_file': int(len(frame))}
    x = frame.loc[frame['age_years'] >= 18].copy()
    flow['adults_18_plus'] = int(len(x))
    x = x.loc[x['reported_diabetes'] == 2].copy()
    flow['no_reported_diabetes'] = int(len(x))
    x = x.loc[x['reported_prediabetes'] == 2].copy()
    flow['no_reported_prediabetes'] = int(len(x))
    x = x.loc[x['hba1c_percent'].notna()].copy()
    flow['valid_hba1c'] = int(len(x))
    x['target'] = (x['hba1c_percent'] >= 5.7).astype('int8')
    x['target_diabetes_range'] = (x['hba1c_percent'] >= 6.5).astype('int8')
    flow['screen_positive'] = int(x['target'].sum())
    return x.reset_index(drop=True), flow

historical_raw, historical_report = load_cycle('2017_mar2020')
temporal_raw, temporal_report = load_cycle('2021_2023')
historical, historical_flow = make_cohort(canonicalize(historical_raw, '2017_mar2020'))
temporal_test, temporal_flow = make_cohort(canonicalize(temporal_raw, '2021_2023'))
del historical_raw, temporal_raw

COHORT_FLOW = {'historical': historical_flow, 'temporal_test': temporal_flow}
print('Historical eligible cohort:', len(historical), '| positives:', int(historical['target'].sum()), '| prevalence:', round(float(historical['target'].mean()), 4))
print('Temporal test eligible cohort:', len(temporal_test), '| positives:', int(temporal_test['target'].sum()), '| prevalence:', round(float(temporal_test['target'].mean()), 4))
print('Cohort flow:', json.dumps(COHORT_FLOW, indent=2))
print('Files loaded historical:', historical_report['loaded_files'])
print('Files loaded temporal:', temporal_report['loaded_files'])
if len(historical) < 200 or len(temporal_test) < 100:
    warnings.warn('Cohort is smaller than expected. Inspect codebooks, source availability and exclusions before interpreting metrics.')
if historical['target'].nunique() < 2 or temporal_test['target'].nunique() < 2:
    raise ValueError('Both outcome classes are required in historical and temporal cohorts; inspect the source data and cohort rules.')

In [ ]:
ONTOLOGY_TTL_EMBEDDED = "@prefix : <https://w3id.org/clinontoscreen/ontology#> .\n@prefix owl: <http://www.w3.org/2002/07/owl#> .\n@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n@prefix dcterms: <http://purl.org/dc/terms/> .\n@prefix loinc: <https://loinc.org/> .\n\n<https://w3id.org/clinontoscreen/ontology> a owl:Ontology ;\n    dcterms:title \"ClinOntoScreen diabetes screening knowledge model\"@en ;\n    dcterms:description \"A bounded, auditable representation of selected clinical screening criteria for research on NHANES. This is not a complete clinical guideline or a diagnostic system.\"@en ;\n    owl:versionInfo \"0.2.0\" ;\n    dcterms:created \"2026-10-29\"^^xsd:date ;\n    dcterms:modified \"2026-10-30\"^^xsd:date ;\n    dcterms:source <https://diabetesjournals.org/care/article/49/Supplement_1/S27/163926/2-Diagnosis-and-Classification-of-Diabetes> .\n\n:Person a owl:Class ; rdfs:label \"Person\"@en .\n:ClinicalMeasurement a owl:Class ; rdfs:label \"Clinical measurement\"@en .\n:RiskFactor a owl:Class ; rdfs:label \"Risk factor\"@en .\n:ScreeningCriterion a owl:Class ; rdfs:label \"Screening criterion\"@en .\n:ScreeningRule a owl:Class ; rdfs:label \"Screening rule\"@en .\n:ScreeningAssessment a owl:Class ; rdfs:label \"Screening assessment\"@en .\n:OutcomeDefinition a owl:Class ; rdfs:label \"Research outcome definition\"@en ;\n    rdfs:comment \"Outcome definitions are used only for evaluation and must not become predictors or screening rules.\"@en .\n:DataVariable a owl:Class ; rdfs:label \"Dataset variable\"@en .\n:GuidelineSource a owl:Class ; rdfs:label \"Guideline source\"@en .\n\n:hasMeasurement a owl:ObjectProperty ; rdfs:domain :Person ; rdfs:range :ClinicalMeasurement .\n:hasRiskFactor a owl:ObjectProperty ; rdfs:domain :Person ; rdfs:range :RiskFactor .\n:usesCriterion a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :ScreeningCriterion .\n:usesVariable a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :DataVariable .\n:derivedFrom a owl:ObjectProperty ; rdfs:domain :DataVariable ; rdfs:range :DataVariable .\n:hasSource a owl:ObjectProperty ; rdfs:range :GuidelineSource .\n:producesAssessment a owl:ObjectProperty ; rdfs:domain :ScreeningRule ; rdfs:range :ScreeningAssessment .\n:meetsCriterion a owl:ObjectProperty ; rdfs:domain :ScreeningAssessment ; rdfs:range :ScreeningCriterion .\n\n:ageYears a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:integer .\n:bmi a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:systolicBloodPressure a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:diastolicBloodPressure a owl:DatatypeProperty ; rdfs:domain :Person ; rdfs:range xsd:decimal .\n:ruleExpression a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:string .\n:thresholdValue a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:thresholdUnit a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:string .\n:requiresRiskFactorCount a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:integer .\n:systolicThreshold a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:diastolicThreshold a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:decimal .\n:categoryCodes a owl:DatatypeProperty ; rdfs:domain :ScreeningCriterion ; rdfs:range xsd:string .\n:implementationNote a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:string .\n:sectionReference a owl:DatatypeProperty ; rdfs:range xsd:string ;\n    rdfs:comment \"Identifier of the guideline section a rule is taken from, stored so every encoded flag is traceable to source text.\"@en .\n:operationalizationStatus a owl:DatatypeProperty ; rdfs:range xsd:string ;\n    rdfs:comment \"Whether, and how, a criterion could be operationalized from the harmonized NHANES public files.\"@en .\n:ruleOrder a owl:DatatypeProperty ; rdfs:domain :ScreeningRule ; rdfs:range xsd:integer .\n:nhanesVariable a owl:DatatypeProperty ; rdfs:domain :DataVariable ; rdfs:range xsd:string .\n:sourceURL a owl:DatatypeProperty ; rdfs:domain :GuidelineSource ; rdfs:range xsd:anyURI .\n\n:ADAStandards2026 a :GuidelineSource ;\n    rdfs:label \"American Diabetes Association Standards of Care in Diabetes\u20142026\"@en ;\n    :sourceURL \"https://diabetesjournals.org/care/article/49/Supplement_1/S27/163926/2-Diagnosis-and-Classification-of-Diabetes\"^^xsd:anyURI ;\n    rdfs:comment \"Selected screening recommendations are operationalized only where the required NHANES variables are available. Refer to the source document for the full recommendation.\"@en .\n\n:Age35Criterion a :ScreeningCriterion ;\n    rdfs:label \"Adult age-based screening criterion\"@en ;\n    rdfs:comment \"ADA guidance recommends screening adults beginning at age 35. This project encodes this as a screening eligibility criterion, not as a diagnosis.\"@en ;\n    :thresholdValue \"35\"^^xsd:decimal ; :thresholdUnit \"years\" .\n\n:BMIOverweightGeneralCriterion a :ScreeningCriterion ;\n    rdfs:label \"BMI threshold for overweight in general screening criteria\"@en ;\n    :thresholdValue \"25\"^^xsd:decimal ; :thresholdUnit \"kg/m2\" ;\n    rdfs:comment \"The selected ADA screening recommendation uses BMI >=25 kg/m2 for adults with overweight or obesity, subject to additional risk criteria when younger than the age-based screening threshold.\"@en .\n\n:BMIOverweightAsianCriterion a :ScreeningCriterion ;\n    rdfs:label \"BMI threshold used for Asian ancestry in screening criteria\"@en ;\n    :thresholdValue \"23\"^^xsd:decimal ; :thresholdUnit \"kg/m2\" ;\n    rdfs:comment \"The selected ADA screening recommendation uses BMI >=23 kg/m2 for individuals of Asian ancestry. NHANES categories are an imperfect proxy for ancestry; this is a documented limitation and must be ablated in evaluation.\"@en .\n\n:HypertensionRiskCriterion a :ScreeningCriterion ;\n    rdfs:label \"Hypertension risk factor\"@en ;\n    :systolicThreshold \"130\"^^xsd:decimal ;\n    :diastolicThreshold \"80\"^^xsd:decimal ;\n    rdfs:comment \"Operationalized from measured blood pressure, self-reported hypertension history, or prescribed hypertension treatment when available in NHANES.\"@en .\n\n:CVDHistoryRiskCriterion a :ScreeningCriterion ;\n    rdfs:label \"Cardiovascular disease history risk factor\"@en ;\n    rdfs:comment \"Operationalized from selected self-reported cardiovascular history items. It is not a comprehensive cardiovascular history.\"@en .\n\n:PhysicalInactivityRiskCriterion a :ScreeningCriterion ;\n    rdfs:label \"Physical inactivity risk factor\"@en ;\n    :thresholdValue \"0\"^^xsd:decimal ; :thresholdUnit \"minutes per week of harmonized leisure-time activity\" ;\n    :operationalizationStatus \"operationalized with cycle-harmonized leisure-time activity minutes: the yes/no work and recreation items (PAQ605/620/650/665) exist only in 2017\u2013March 2020, so leisure-time moderate and vigorous activity minutes are used instead (PAD660+PAD675 with skip-pattern zeros in 2017\u2013March 2020; PAD800+PAD820 with skip-pattern zeros in August 2021\u2013August 2023)\" ;\n    rdfs:comment \"A person is recorded as inactive when harmonized leisure-time activity minutes are zero; when minutes are unknown the criterion is not assessable and is never treated as absent.\"@en .\n\n:HigherRiskEthnicityCriterion a :ScreeningCriterion ;\n    rdfs:label \"Higher-risk race, ethnicity, or ancestry factor in screening guidance\"@en ;\n    :categoryCodes \"1,2,4,6\" ;\n    rdfs:comment \"Encodes only an approximate grouping supported by available NHANES categories. This is included as a guideline criterion for a sensitivity analysis, not interpreted as biological causation. The primary paper must discuss fairness and show an ablation without this feature.\"@en .\n\n:AgeBasedScreeningRule a :ScreeningRule ;\n    rdfs:label \"Age-based screening rule\"@en ;\n    :ruleOrder 1 ;\n    :ruleExpression \"age_years >= 35\" ;\n    :usesCriterion :Age35Criterion ;\n    :usesVariable :AgeVariable ;\n    :hasSource :ADAStandards2026 ;\n    :sectionReference \"ADA Standards of Care in Diabetes\u20142026, Chapter 2 (Diagnosis and Classification of Diabetes), screening recommendations for adults\" ;\n    :operationalizationStatus \"operationalized: RIDAGEYR is present in both cycles\" ;\n    :implementationNote \"If age is known and >=35, mark screening eligibility. This is not a diagnosis and does not imply confirmed disease.\"@en .\n\n:YoungerAdultRiskBasedRule a :ScreeningRule ;\n    rdfs:label \"Younger-adult BMI plus additional risk-factor rule\"@en ;\n    :ruleOrder 2 ;\n    :ruleExpression \"age_years < 35 AND bmi_meets_ancestry_threshold AND at_least_one_observed_additional_risk_factor\" ;\n    :requiresRiskFactorCount 1 ;\n    :usesCriterion :BMIOverweightGeneralCriterion, :BMIOverweightAsianCriterion, :HypertensionRiskCriterion, :CVDHistoryRiskCriterion, :PhysicalInactivityRiskCriterion, :HigherRiskEthnicityCriterion ;\n    :usesVariable :AgeVariable, :BMIVariable, :RaceEthnicityVariable, :SystolicBPVariable, :DiastolicBPVariable, :HypertensionHistoryVariable, :HypertensionTreatmentVariable, :CVDHistoryVariable, :PhysicalActivityVariable ;\n    :hasSource :ADAStandards2026 ;\n    :sectionReference \"ADA Standards of Care in Diabetes\u20142026, Chapter 2 (Diagnosis and Classification of Diabetes), screening recommendations for younger adults with overweight/obesity and additional risk factors\" ;\n    :operationalizationStatus \"partially operationalized: family history of diabetes, gestational diabetes and PCOS items are absent from one or both cycles; see :FamilyHistoryCriterion, :GestationalDiabetesCriterion and :PCOSCriterion\" ;\n    :implementationNote \"This is a bounded operationalization of selected screening factors. Missing criteria are marked unknown; they are not treated as absent. The complete guideline contains other factors not represented here.\"@en .\n\n:FamilyHistoryCriterion a :ScreeningCriterion ;\n    rdfs:label \"First-degree relative with diabetes\"@en ;\n    :operationalizationStatus \"not operationalized: MCQ300C (close relative had diabetes) is present in the 2017\u2013March 2020 public MCQ file but is not included in the August 2021\u2013August 2023 public MCQ file, so it cannot be harmonized across the temporal split\" ;\n    rdfs:comment \"Listed for completeness and traceability only. It is not evaluated by the rule executor and is not a model feature.\"@en .\n\n:GestationalDiabetesCriterion a :ScreeningCriterion ;\n    rdfs:label \"History of gestational diabetes\"@en ;\n    :operationalizationStatus \"not operationalized: no gestational diabetes item is available in the selected NHANES public files\" ;\n    rdfs:comment \"Listed for completeness and traceability only. It is not evaluated by the rule executor and is not a model feature.\"@en .\n\n:PCOSCriterion a :ScreeningCriterion ;\n    rdfs:label \"Polycystic ovary syndrome history\"@en ;\n    :operationalizationStatus \"not operationalized: no PCOS item is available in the selected NHANES public files\" ;\n    rdfs:comment \"Listed for completeness and traceability only. It is not evaluated by the rule executor and is not a model feature.\"@en .\n\n:AgeVariable a :DataVariable ; rdfs:label \"Age in years\"@en ; :nhanesVariable \"RIDAGEYR\" .\n:SexVariable a :DataVariable ; rdfs:label \"NHANES sex/gender field (as coded in source)\"@en ; :nhanesVariable \"RIAGENDR\" .\n:RaceEthnicityVariable a :DataVariable ; rdfs:label \"NHANES race/Hispanic origin including non-Hispanic Asian category\"@en ; :nhanesVariable \"RIDRETH3\" .\n:BMIVariable a :DataVariable ; rdfs:label \"Body mass index\"@en ; :nhanesVariable \"BMXBMI\" .\n:WaistVariable a :DataVariable ; rdfs:label \"Waist circumference\"@en ; :nhanesVariable \"BMXWAIST\" .\n:SystolicBPVariable a :DataVariable ; rdfs:label \"Oscillometric systolic blood pressure reading\"@en ; :nhanesVariable \"BPXOSY1-BPXOSY4\" .\n:DiastolicBPVariable a :DataVariable ; rdfs:label \"Oscillometric diastolic blood pressure reading\"@en ; :nhanesVariable \"BPXODI1-BPXODI4\" .\n:HypertensionHistoryVariable a :DataVariable ; rdfs:label \"Ever told by a health professional that blood pressure was high\"@en ; :nhanesVariable \"BPQ020\" .\n:HypertensionTreatmentVariable a :DataVariable ; rdfs:label \"Currently taking prescribed medicine for hypertension\"@en ; :nhanesVariable \"BPQ040A\" ;\n    :operationalizationStatus \"cycle-asymmetric: BPQ040A exists in the 2017\u2013March 2020 public BPQ file; the August 2021\u2013August 2023 public BPQ file contains no hypertension-treatment item, so this column is unobserved in the later cycle and is treated as unknown, not as 'no'\" .\n:CVDHistoryVariable a :DataVariable ; rdfs:label \"Selected cardiovascular history questions\"@en ; :nhanesVariable \"MCQ160B-MCQ160F\" .\n:PhysicalActivityVariable a :DataVariable ; rdfs:label \"Harmonized leisure-time physical activity minutes and daily sedentary minutes\"@en ;\n    :nhanesVariable \"PAD660+PAD675 (2017\u2013March 2020); PAD800+PAD820 (August 2021\u2013August 2023); PAD680 sedentary minutes in both cycles\" ;\n    :operationalizationStatus \"harmonized: the PAQ605/620/650/665 yes/no items are not present in the August 2021\u2013August 2023 public file and are excluded from the model schema\" .\n:SmokingVariable a :DataVariable ; rdfs:label \"Ever smoked at least 100 cigarettes\"@en ; :nhanesVariable \"SMQ020\" .\n:FamilyHistoryVariable a :DataVariable ; rdfs:label \"Close biological relative told they had diabetes\"@en ; :nhanesVariable \"MCQ300C\" ;\n    :operationalizationStatus \"not harmonized: present in the 2017\u2013March 2020 public MCQ file, not included in the August 2021\u2013August 2023 public MCQ file\" .\n:EducationVariable a :DataVariable ; rdfs:label \"Education level for adults\"@en ; :nhanesVariable \"DMDEDUC2\" .\n:IncomeVariable a :DataVariable ; rdfs:label \"Family income-to-poverty ratio\"@en ; :nhanesVariable \"INDFMPIR\" .\n\n:HbA1cTargetDefinition a :OutcomeDefinition ;\n    rdfs:label \"Research outcome: HbA1c-defined screen-positive dysglycemia\"@en ;\n    rdfs:comment \"Among adults with valid HbA1c who report neither prior diabetes nor prior prediabetes, target=1 when LBXGH >=5.7%. This is a research label based on one survey measurement, not a confirmed clinical diagnosis.\"@en ;\n    :thresholdValue \"5.7\"^^xsd:decimal ; :thresholdUnit \"percent HbA1c\" ;\n    :usesVariable :HbA1cVariable ;\n    :sectionReference \"ADA Standards of Care in Diabetes\u20142026, Chapter 2 (Diagnosis and Classification of Diabetes), diagnostic criteria for prediabetes\" ;\n    :hasSource :ADAStandards2026 .\n\n:DiabetesRangeSecondaryOutcome a :OutcomeDefinition ;\n    rdfs:label \"Secondary outcome: single HbA1c measurement in diabetes range\"@en ;\n    rdfs:comment \"Among the same eligible cohort, target=1 when LBXGH >=6.5%. A single measurement is not by itself a confirmed diagnosis.\"@en ;\n    :thresholdValue \"6.5\"^^xsd:decimal ; :thresholdUnit \"percent HbA1c\" ;\n    :usesVariable :HbA1cVariable ;\n    :sectionReference \"ADA Standards of Care in Diabetes\u20142026, Chapter 2 (Diagnosis and Classification of Diabetes), diagnostic criteria for diabetes\" ;\n    :hasSource :ADAStandards2026 .\n\n:HDLCholesterolVariable a :DataVariable ;\n    rdfs:label \"Direct high-density lipoprotein cholesterol measurement\"@en ;\n    rdfs:comment \"Non-glycemic laboratory predictor published for both cycles (HDL file). It is not part of the encoded screening rules.\"@en ;\n    :nhanesVariable \"LBDHDD\" .\n\n:HbA1cVariable a :DataVariable ;\n    rdfs:label \"NHANES glycohemoglobin (HbA1c) laboratory measurement\"@en ;\n    :nhanesVariable \"LBXGH\" ;\n    rdfs:comment \"Outcome-only variable. Strictly prohibited from the ML feature matrix.\"@en ;\n    :derivedFrom loinc:4548-4 .\n\nloinc:4548-4 rdfs:label \"Hemoglobin A1c/Hemoglobin.total in Blood\"@en .\n"

ontology_candidates = [Path.cwd() / 'ontology' / 'diabetes_screening.ttl', Path.cwd().parent / 'ontology' / 'diabetes_screening.ttl']
ontology_path = next((p for p in ontology_candidates if p.exists()), None)
ontology_graph = Graph()
if ontology_path:
    ontology_graph.parse(ontology_path, format='turtle')
    print('Loaded repository ontology:', ontology_path)
else:
    ontology_graph.parse(data=ONTOLOGY_TTL_EMBEDDED, format='turtle')
    ontology_path = ARTIFACT_DIR / 'diabetes_screening.ttl'
    ontology_path.write_text(ONTOLOGY_TTL_EMBEDDED, encoding='utf-8')
    print('Loaded embedded ontology and wrote a copy to:', ontology_path)

EX_NS = 'https://w3id.org/clinontoscreen/ontology#'
def ontology_parameter(subject_name, predicate_name, default):
    from rdflib import URIRef
    subject = URIRef(EX_NS + subject_name)
    predicate = URIRef(EX_NS + predicate_name)
    values = list(ontology_graph.objects(subject, predicate))
    return float(values[0]) if values else float(default)
def ontology_text(subject_name, predicate_name, default=''):
    from rdflib import URIRef
    values = list(ontology_graph.objects(URIRef(EX_NS + subject_name), URIRef(EX_NS + predicate_name)))
    return str(values[0]) if values else default
AGE_THRESHOLD = ontology_parameter('Age35Criterion', 'thresholdValue', 35)
BMI_GENERAL = ontology_parameter('BMIOverweightGeneralCriterion', 'thresholdValue', 25)
BMI_ASIAN = ontology_parameter('BMIOverweightAsianCriterion', 'thresholdValue', 23)
SBP_THRESHOLD = ontology_parameter('HypertensionRiskCriterion', 'systolicThreshold', 130)
DBP_THRESHOLD = ontology_parameter('HypertensionRiskCriterion', 'diastolicThreshold', 80)
INACTIVITY_MAX_MINUTES = ontology_parameter('PhysicalInactivityRiskCriterion', 'thresholdValue', 0)
from rdflib import URIRef
_codes = list(ontology_graph.objects(URIRef(EX_NS+'HigherRiskEthnicityCriterion'), URIRef(EX_NS+'categoryCodes')))
HIGHER_RISK_ETHNICITY_CODES = [int(x.strip()) for x in (str(_codes[0]) if _codes else '1,2,4,6').split(',')]
RULE_SECTIONS = {rule: ontology_text(rule, 'sectionReference', 'unspecified') for rule in ['AgeBasedScreeningRule','YoungerAdultRiskBasedRule']}
print('Ontology triples:', len(ontology_graph), '| thresholds:', AGE_THRESHOLD, BMI_GENERAL, BMI_ASIAN, SBP_THRESHOLD, DBP_THRESHOLD)
for rule, section in RULE_SECTIONS.items():
    print(f'  {rule} -> {section}')

RAW_NUMERIC = ['age_years','income_poverty_ratio','bmi','waist_cm','hdl_chol_mgdl','leisure_activity_minutes','sedentary_minutes','systolic_bp_mean','diastolic_bp_mean']
RAW_CATEGORICAL = ['sex_code','race_ethnicity_code','education_code','hypertension_history','hypertension_treatment','ever_smoked_100_cigs','heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']
RAW_FEATURES = RAW_NUMERIC + RAW_CATEGORICAL
ONTOLOGY_FEATURES = ['onto_age_based_screening','onto_bmi_threshold_met','onto_hypertension_risk','onto_cvd_history','onto_physical_inactivity','onto_higher_risk_ethnicity','onto_additional_risk_factor_count','onto_screening_eligible','onto_criteria_observed_count','onto_criteria_missing_count','onto_rule_priority_score']
MANUAL_FEATURES = [c.replace('onto_', 'manual_') for c in ONTOLOGY_FEATURES]
FORBIDDEN_FEATURE_TOKENS = ('hba1c','glucose','reported_diabetes','reported_prediabetes','target','diabetes_medication')
assert not any(any(token in c.lower() for token in FORBIDDEN_FEATURE_TOKENS) for c in RAW_FEATURES + ONTOLOGY_FEATURES)
assert 'SEQN' not in RAW_FEATURES
assert all(len(section) > 20 for section in RULE_SECTIONS.values()), 'Every executable rule must carry a guideline section reference'

def yes_no_flag(frame, col):
    vals = pd.to_numeric(frame[col], errors='coerce') if col in frame else pd.Series(np.nan,index=frame.index,dtype='float64')
    return vals.map({1.0:1.0,2.0:0.0}).astype('float64')

def any_flag(flags, index):
    values = pd.concat(flags, axis=1)
    result = pd.Series(np.nan,index=index,dtype='float64')
    any_yes = values.eq(1).any(axis=1); any_known = values.notna().any(axis=1)
    result.loc[any_yes] = 1.0; result.loc[~any_yes & any_known] = 0.0
    return result

def derive_ontology_features(frame, include_ethnicity=True):
    out = frame.copy()
    age = pd.to_numeric(out.get('age_years',pd.Series(np.nan,index=out.index)),errors='coerce')
    bmi = pd.to_numeric(out.get('bmi',pd.Series(np.nan,index=out.index)),errors='coerce')
    race = pd.to_numeric(out.get('race_ethnicity_code',pd.Series(np.nan,index=out.index)),errors='coerce')
    asian = race.eq(6); cutoff = pd.Series(BMI_GENERAL,index=out.index,dtype='float64'); cutoff.loc[asian] = BMI_ASIAN
    bmi_flag = pd.Series(np.nan,index=out.index,dtype='float64'); known_bmi = bmi.notna() & race.notna(); bmi_flag.loc[known_bmi] = (bmi.loc[known_bmi] >= cutoff.loc[known_bmi]).astype(float)
    age_flag = pd.Series(np.nan,index=out.index,dtype='float64'); age_flag.loc[age.notna()] = (age.loc[age.notna()] >= AGE_THRESHOLD).astype(float)
    sbp = pd.to_numeric(out.get('systolic_bp_mean',pd.Series(np.nan,index=out.index)),errors='coerce')
    dbp = pd.to_numeric(out.get('diastolic_bp_mean',pd.Series(np.nan,index=out.index)),errors='coerce')
    measured_bp = pd.Series(np.nan,index=out.index,dtype='float64'); bp_known = sbp.notna() | dbp.notna()
    measured_bp.loc[bp_known] = (((sbp >= SBP_THRESHOLD) | (dbp >= DBP_THRESHOLD)).loc[bp_known]).astype(float)
    hypertension = any_flag([measured_bp,yes_no_flag(out,'hypertension_history'),yes_no_flag(out,'hypertension_treatment')],out.index)
    cvd = any_flag([yes_no_flag(out,c) for c in ['heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']],out.index)
    minutes = pd.to_numeric(out.get('leisure_activity_minutes',pd.Series(np.nan,index=out.index)),errors='coerce')
    inactivity = pd.Series(np.nan,index=out.index,dtype='float64')
    inactivity.loc[minutes.notna() & (minutes <= INACTIVITY_MAX_MINUTES)] = 1.0
    inactivity.loc[minutes.notna() & (minutes > INACTIVITY_MAX_MINUTES)] = 0.0
    ethnicity = pd.Series(np.nan,index=out.index,dtype='float64')
    if include_ethnicity:
        ethnicity.loc[race.notna()] = race.loc[race.notna()].isin(HIGHER_RISK_ETHNICITY_CODES).astype(float)
    else:
        ethnicity = pd.Series(0.0,index=out.index,dtype='float64')
    risk_columns = [hypertension,cvd,inactivity]
    if include_ethnicity: risk_columns.append(ethnicity)
    risks = pd.concat(risk_columns,axis=1); risk_count = risks.fillna(0).sum(axis=1).astype(float); risk_count.loc[risks.notna().sum(axis=1)==0] = np.nan
    eligible = pd.Series(np.nan,index=out.index,dtype='float64'); eligible.loc[age >= AGE_THRESHOLD] = 1.0
    younger = age.notna() & (age < AGE_THRESHOLD) & bmi_flag.notna(); eligible.loc[younger & bmi_flag.eq(0)] = 0.0
    eligible.loc[younger & bmi_flag.eq(1) & (risk_count >= 1)] = 1.0; eligible.loc[younger & bmi_flag.eq(1) & risk_count.eq(0)] = 0.0
    criteria_columns = [age_flag,bmi_flag,hypertension,cvd,inactivity]
    if include_ethnicity: criteria_columns.append(ethnicity)
    criteria = pd.concat(criteria_columns,axis=1); observed = criteria.notna().sum(axis=1).astype(float); missing = criteria.isna().sum(axis=1).astype(float)
    priority = criteria.fillna(0).sum(axis=1).astype(float) / observed.replace(0,np.nan)
    vals = [age_flag,bmi_flag,hypertension,cvd,inactivity,ethnicity,risk_count,eligible,observed,missing,priority.fillna(0.0)]
    for name,val in zip(ONTOLOGY_FEATURES,vals): out[name]=val
    return out

def derive_manual_control(frame):
    out = frame.copy(); age=pd.to_numeric(out['age_years'],errors='coerce'); bmi=pd.to_numeric(out['bmi'],errors='coerce'); race=pd.to_numeric(out['race_ethnicity_code'],errors='coerce')
    threshold=np.where(race.eq(6),BMI_ASIAN,BMI_GENERAL); bmi_flag=pd.Series(np.where(bmi.notna()&race.notna(),(bmi>=threshold).astype(float),np.nan),index=out.index)
    age_flag=pd.Series(np.where(age.notna(),(age>=AGE_THRESHOLD).astype(float),np.nan),index=out.index)
    sbp=pd.to_numeric(out['systolic_bp_mean'],errors='coerce'); dbp=pd.to_numeric(out['diastolic_bp_mean'],errors='coerce')
    measured=pd.Series(np.where(sbp.notna()|dbp.notna(),((sbp>=SBP_THRESHOLD)|(dbp>=DBP_THRESHOLD)).astype(float),np.nan),index=out.index)
    def yn(col): return pd.to_numeric(out[col],errors='coerce').map({1.0:1.0,2.0:0.0}).astype('float64')
    def or_flags(items):
        z=pd.concat(items,axis=1); a=z.eq(1).any(axis=1); known=z.notna().any(axis=1); r=pd.Series(np.nan,index=out.index,dtype='float64'); r.loc[a]=1.0; r.loc[~a&known]=0.0; return r
    hypertension=or_flags([measured,yn('hypertension_history'),yn('hypertension_treatment')])
    cvd=or_flags([yn(c) for c in ['heart_failure_history','coronary_heart_disease_history','angina_history','heart_attack_history','stroke_history']])
    mins=pd.to_numeric(out['leisure_activity_minutes'],errors='coerce')
    inactive=pd.Series(np.nan,index=out.index,dtype='float64'); inactive.loc[mins.notna()&(mins<=INACTIVITY_MAX_MINUTES)]=1.; inactive.loc[mins.notna()&(mins>INACTIVITY_MAX_MINUTES)]=0.
    ethnicity=pd.Series(np.nan,index=out.index,dtype='float64'); ethnicity.loc[race.notna()]=race.loc[race.notna()].isin(HIGHER_RISK_ETHNICITY_CODES).astype(float)
    risks=pd.concat([hypertension,cvd,inactive,ethnicity],axis=1); count=risks.fillna(0).sum(axis=1).astype(float); count.loc[risks.notna().sum(axis=1)==0]=np.nan
    eligible=pd.Series(np.nan,index=out.index,dtype='float64'); eligible.loc[age>=AGE_THRESHOLD]=1.; younger=age.notna()&(age<AGE_THRESHOLD)&bmi_flag.notna(); eligible.loc[younger&bmi_flag.eq(0)]=0.; eligible.loc[younger&bmi_flag.eq(1)&(count>=1)]=1.; eligible.loc[younger&bmi_flag.eq(1)&count.eq(0)]=0.
    criteria=pd.concat([age_flag,bmi_flag,hypertension,cvd,inactive,ethnicity],axis=1); observed=criteria.notna().sum(axis=1).astype(float); missing=criteria.isna().sum(axis=1).astype(float); priority=criteria.fillna(0).sum(axis=1).astype(float)/observed.replace(0,np.nan)
    vals=[age_flag,bmi_flag,hypertension,cvd,inactive,ethnicity,count,eligible,observed,missing,priority.fillna(0.)]
    for name,val in zip(MANUAL_FEATURES,vals): out[name]=val
    return out

def cdc_ada_risk_score(frame):
    age = pd.to_numeric(frame.get('age_years', pd.Series(dtype='float64')), errors='coerce')
    score = pd.Series(0.0, index=frame.index, dtype='float64')
    score += np.where(age >= 60, 3, np.where(age >= 50, 2, np.where(age >= 40, 1, 0)))
    sex = pd.to_numeric(frame.get('sex_code', pd.Series(dtype='float64')), errors='coerce')
    score += (sex == 1).astype(float)
    bp_hist = pd.to_numeric(frame.get('hypertension_history', pd.Series(dtype='float64')), errors='coerce')
    score += (bp_hist == 1).astype(float)
    minutes = pd.to_numeric(frame.get('leisure_activity_minutes', pd.Series(dtype='float64')), errors='coerce')
    score += (minutes.notna() & (minutes <= INACTIVITY_MAX_MINUTES)).astype(float)
    bmi = pd.to_numeric(frame.get('bmi', pd.Series(dtype='float64')), errors='coerce')
    race = pd.to_numeric(frame.get('race_ethnicity_code', pd.Series(dtype='float64')), errors='coerce')
    overweight_cut = np.where(race == 6, 23.0, 25.0)
    score += np.where(bmi >= 40, 3, np.where(bmi >= 30, 2, np.where(bmi >= overweight_cut, 1, 0)))
    return score

CDC_ADA_UNAVAILABLE_ITEMS = ['gestational_diabetes_history', 'family_diabetes_history']
rule_base='https://w3id.org/clinontoscreen/ontology#'
for rn in ['AgeBasedScreeningRule','YoungerAdultRiskBasedRule']:
    r=URIRef(rule_base+rn)
    assert list(ontology_graph.objects(r,URIRef(rule_base+'hasSource'))), f'{rn} has no source'
    assert list(ontology_graph.objects(r,URIRef(rule_base+'ruleExpression'))), f'{rn} has no expression'
    assert list(ontology_graph.objects(r,URIRef(rule_base+'sectionReference'))), f'{rn} has no section reference'
print('Ontology source, expression and section-reference checks passed.')

In [ ]:
# Metric helpers. Capacity metrics report the prevalence ceiling required by plan addendum §4.
def metric_pack(y, p):
    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float)
    return {'n':int(len(y)),'positive_count':int(y.sum()),'prevalence':float(y.mean()),
            'auroc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'auprc':float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'brier':float(brier_score_loss(y,p))}

def capacity_pack(y, score, capacity):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float)
    n=max(1,int(np.ceil(len(y)*capacity)))
    idx=np.argsort(-np.nan_to_num(score,nan=-np.inf),kind='mergesort')[:n]
    found=int(y[idx].sum()); pos=int(y.sum())
    prevalence=float(y.mean()) if len(y) else float('nan')
    ceiling=min(1.0, capacity/prevalence) if prevalence>0 else float('nan')
    sensitivity=float(found/pos) if pos else float('nan')
    ppv=float(found/n)
    return {'capacity':float(capacity),'n_total':int(len(y)),'n_selected':int(n),'positive_total':pos,'positive_found':found,
            'prevalence':prevalence,'sensitivity':sensitivity,'sensitivity_ceiling':ceiling,
            'fraction_of_ceiling':float(sensitivity/ceiling) if ceiling>0 else float('nan'),
            'precision':ppv,'ppv':ppv,'lift':float(ppv/prevalence) if prevalence>0 else float('nan'),
            'false_referrals':int(n-found)}

def paired_capacity_bootstrap(y, score_a, score_b, capacity=PRIMARY_CAPACITY, n_bootstrap=N_BOOTSTRAP, seed=2026):
    """Participant bootstrap for sensitivity(score_a) - sensitivity(score_b) at a fixed capacity."""
    y=np.asarray(y,dtype=int); sa=np.asarray(score_a,dtype=float); sb=np.asarray(score_b,dtype=float)
    rng=np.random.default_rng(seed); diffs=[]; attempts=0
    while len(diffs)<n_bootstrap and attempts < n_bootstrap*3:
        attempts += 1
        idx=rng.integers(0,len(y),size=len(y))
        yb=y[idx]
        if yb.sum()==0: continue
        n=max(1,int(np.ceil(len(yb)*capacity)))
        a=float(yb[np.argsort(-np.nan_to_num(sa[idx],nan=-np.inf),kind='mergesort')[:n]].sum()/yb.sum())
        b=float(yb[np.argsort(-np.nan_to_num(sb[idx],nan=-np.inf),kind='mergesort')[:n]].sum()/yb.sum())
        diffs.append(a-b)
    point_a=capacity_pack(y,sa,capacity)['sensitivity']; point_b=capacity_pack(y,sb,capacity)['sensitivity']
    if not diffs:
        return {'difference':point_a-point_b,'ci_low':float('nan'),'ci_high':float('nan'),'std_error':float('nan'),'n_bootstrap':0}
    arr=np.array(diffs)
    return {'difference':float(point_a-point_b),'ci_low':float(np.quantile(arr,.025)),'ci_high':float(np.quantile(arr,.975)),
            'std_error':float(np.std(arr,ddof=1)),'n_bootstrap':len(diffs)}

def paired_metric_bootstrap(y, p_a, p_b, metric='auprc', n_bootstrap=N_BOOTSTRAP, seed=2026):
    y=np.asarray(y,dtype=int); pa=np.asarray(p_a,dtype=float); pb=np.asarray(p_b,dtype=float)
    fn = average_precision_score if metric=='auprc' else roc_auc_score
    rng=np.random.default_rng(seed); diffs=[]; attempts=0
    while len(diffs)<n_bootstrap and attempts<n_bootstrap*3:
        attempts+=1
        idx=rng.integers(0,len(y),size=len(y))
        if len(np.unique(y[idx]))<2: continue
        diffs.append(float(fn(y[idx],pa[idx])-fn(y[idx],pb[idx])))
    arr=np.array(diffs) if diffs else np.array([np.nan])
    return {'metric':metric,'difference':float(fn(y,pa)-fn(y,pb)),
            'ci_low':float(np.nanquantile(arr,.025)),'ci_high':float(np.nanquantile(arr,.975)),
            'std_error':float(np.nanstd(arr,ddof=1)),'n_bootstrap':int(len(diffs))}

# Data quality, cohort flow, missingness and leakage contract.
for label, df in [('historical', historical), ('temporal_test', temporal_test)]:
    print(f"\n{label}: n={len(df):,}; positives={int(df['target'].sum()):,}; prevalence={df['target'].mean():.3f}; diabetes-range secondary positives={int(df['target_diabetes_range'].sum()):,}")
    display(df[RAW_FEATURES].isna().mean().sort_values(ascending=False).rename('missing_fraction').to_frame().head(12))

# Cycle-level availability of every requested source variable, so harmonization gaps are visible.
availability_rows=[]
for period, report in [('historical', historical_report), ('temporal_test', temporal_report)]:
    for stem, missing in report['missing_variables'].items():
        for var in FILE_VARIABLES[stem]:
            availability_rows.append({'period':period,'file':stem,'variable':var,'available':var not in missing})
availability_df = pd.DataFrame(availability_rows).pivot_table(index='variable', columns='period', values='available', aggfunc='first')
availability_df = availability_df.reindex(columns=['historical', 'temporal_test']).fillna(False)
availability_df['harmonized'] = availability_df[['historical', 'temporal_test']].all(axis=1)
print('\nSource-variable availability by period (variables not in both periods are excluded from the model schema):')
display(availability_df.sort_values('harmonized'))
unharmonized = sorted(availability_df.index[~availability_df['harmonized']].astype(str))

capacity_ceiling = {}
for capacity in CAPACITIES:
    ceiling = min(1.0, capacity / float(historical['target'].mean()))
    capacity_ceiling[capacity] = {'historical_prevalence':float(historical['target'].mean()), 'sensitivity_ceiling':ceiling}
print('\nHistorical prevalence and the maximum sensitivity each capacity can reach:')
print(pd.DataFrame(capacity_ceiling).T)
if float(historical['target'].mean()) > PRIMARY_CAPACITY:
    print(f'Historical prevalence ({historical["target"].mean():.3f}) exceeds the {PRIMARY_CAPACITY:.0%} budget, so the primary endpoint is reported as the fraction of the {capacity_ceiling[PRIMARY_CAPACITY]["sensitivity_ceiling"]:.3f} ceiling achieved.')

cohort_qc = {
    'historical_n': int(len(historical)), 'historical_positive_count': int(historical['target'].sum()),
    'historical_prevalence': float(historical['target'].mean()),
    'temporal_test_n': int(len(temporal_test)), 'temporal_test_positive_count': int(temporal_test['target'].sum()),
    'temporal_test_prevalence': float(temporal_test['target'].mean()),
    'cohort_flow': COHORT_FLOW,
    'capacity_ceiling': capacity_ceiling,
    'unharmonized_variables': unharmonized,
    'historical_missing_fraction': historical[RAW_FEATURES].isna().mean().to_dict(),
    'temporal_missing_fraction': temporal_test[RAW_FEATURES].isna().mean().to_dict(),
    'historical_source_report': historical_report, 'temporal_source_report': temporal_report,
}
(ARTIFACT_DIR / 'data_qc.json').write_text(json.dumps(cohort_qc, indent=2, default=str), encoding='utf-8')

# Descriptive composition by period (selection and measurement caveats, plan addendum §7).
def composition(df, label):
    age = pd.to_numeric(df['age_years'], errors='coerce')
    return {
        'period': label, 'n': int(len(df)),
        'mean_age': float(age.mean()), 'pct_age_18_39': float((age<40).mean()), 'pct_age_40_59': float(((age>=40)&(age<60)).mean()), 'pct_age_60_plus': float((age>=60).mean()),
        'pct_male': float((pd.to_numeric(df['sex_code'],errors='coerce')==1).mean()),
        'pct_nh_black': float((pd.to_numeric(df['race_ethnicity_code'],errors='coerce')==4).mean()),
        'pct_nh_asian': float((pd.to_numeric(df['race_ethnicity_code'],errors='coerce')==6).mean()),
        'pct_hispanic': float(pd.to_numeric(df['race_ethnicity_code'],errors='coerce').isin([1,2]).mean()),
        'prevalence': float(df['target'].mean()),
    }
composition_df = pd.DataFrame([composition(historical,'historical'), composition(temporal_test,'temporal_test')])
print('\nCohort composition by period:')
display(composition_df)
composition_df.to_csv(ARTIFACT_DIR/'cohort_composition.csv', index=False)

for col in RAW_FEATURES:
    assert col not in {'SEQN','target','target_diabetes_range','hba1c_percent','reported_diabetes','reported_prediabetes'}
assert not any(any(token in col.lower() for token in FORBIDDEN_FEATURE_TOKENS) for col in RAW_FEATURES)
assert 'SEQN' not in RAW_FEATURES
print('Leakage contract passed. The cohort labels come from HbA1c; target-source fields are not model features.')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(['Historical','Temporal test'], [historical['target'].mean(), temporal_test['target'].mean()])
axes[0].set_title('Primary outcome prevalence'); axes[0].set_ylabel('Fraction positive'); axes[0].set_ylim(0,1)
axes[1].hist(historical['age_years'].dropna(), bins=20, alpha=0.6, label='Historical')
axes[1].hist(temporal_test['age_years'].dropna(), bins=20, alpha=0.6, label='Temporal test')
axes[1].set_title('Age distribution'); axes[1].set_xlabel('Age (NHANES top-codes 80+)'); axes[1].legend()
flow_labels = list(COHORT_FLOW['historical'].keys())
axes[2].plot(flow_labels, [COHORT_FLOW['historical'][k] for k in flow_labels], marker='o', label='Historical')
axes[2].plot(flow_labels, [COHORT_FLOW['temporal_test'][k] for k in flow_labels], marker='o', label='Temporal test')
axes[2].set_title('Cohort flow'); axes[2].tick_params(axis='x', rotation=45); axes[2].legend(); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Model pipelines. All transforms are fitted inside each pipeline, so nothing is learned from the test cycle.
def one_hot_encoder():
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=True)

def make_preprocessor(features):
    numeric_cols = [c for c in features if c in RAW_NUMERIC or c.startswith('onto_') or c.startswith('manual_')]
    categorical_cols = [c for c in features if c not in numeric_cols]
    steps=[]
    if numeric_cols:
        steps.append(('numeric', Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]), numeric_cols))
    if categorical_cols:
        steps.append(('categorical', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',one_hot_encoder())]), categorical_cols))
    return ColumnTransformer(steps, remainder='drop', verbose_feature_names_out=False)

def make_pipeline(model_name, features, seed=42, params=None):
    params = dict(params or {})
    if model_name == 'logistic':
        base = {'C':1.0, 'max_iter':3000, 'solver':'lbfgs', 'random_state':seed}; base.update(params)
        estimator = LogisticRegression(**base)
    elif model_name == 'xgboost':
        base = {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.85,'colsample_bytree':0.85,'reg_lambda':1.0,'min_child_weight':3,'eval_metric':'logloss','n_jobs':2,'random_state':seed,'tree_method':'hist'}; base.update(params)
        estimator = XGBClassifier(**base)
    else:
        raise ValueError(model_name)
    return Pipeline([('preprocess',make_preprocessor(features)),('model',estimator)])

def predict_prob(model, X): return np.asarray(model.predict_proba(X)[:,1],dtype=float)

# Independent historical validation is used for model/policy selection. The later cycle remains locked.
hist_raw = historical[RAW_FEATURES].copy()
hist_onto = derive_ontology_features(hist_raw)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
hist_onto_no_ethnicity = derive_ontology_features(hist_raw, include_ethnicity=False)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
hist_manual = derive_manual_control(hist_raw)[RAW_FEATURES + MANUAL_FEATURES].copy()
# Exact equivalence of values between ontology-derived and independently coded control features is expected.
for of, mf in zip(ONTOLOGY_FEATURES, MANUAL_FEATURES):
    a = hist_onto[of].to_numpy(dtype=float); b = hist_manual[mf].to_numpy(dtype=float)
    assert np.allclose(a, b, equal_nan=True), f'Feature control mismatch: {of} vs {mf}'
print('Generic feature-engineering control matches the same rule values independently.')
print('Ontology rule coverage on historical data:',
      {'eligible': float(hist_onto['onto_screening_eligible'].mean()),
       'criteria_missing_mean': float(hist_onto['onto_criteria_missing_count'].mean()),
       'fully_assessable': float((hist_onto['onto_criteria_missing_count']==0).mean())})

train_idx, val_idx = train_test_split(np.arange(len(historical)), test_size=0.20, random_state=42, stratify=historical['target'])
y_all = historical['target'].astype(int).to_numpy()
y_train, y_val = y_all[train_idx], y_all[val_idx]
X_variants = {
    'logistic_raw': (hist_raw, RAW_FEATURES, 'logistic'),
    'xgboost_raw': (hist_raw, RAW_FEATURES, 'xgboost'),
    'logistic_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'logistic'),
    'xgboost_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_ontology_no_ethnicity': (hist_onto_no_ethnicity, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_generic_control': (hist_manual, RAW_FEATURES+MANUAL_FEATURES, 'xgboost'),
}
param_grids = {
    'logistic': {'model__C':[0.1,1.0,10.0]},
    'xgboost': {'model__n_estimators':[150,300], 'model__max_depth':[2,3], 'model__learning_rate':[0.03,0.07]},
}
searches, best_params, validation_probabilities = {}, {}, {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for name, (X_all_variant, features, family) in X_variants.items():
    X_train = X_all_variant.iloc[train_idx][features]
    X_val = X_all_variant.iloc[val_idx][features]
    search = GridSearchCV(make_pipeline(family,features,seed=42), param_grids[family], scoring='average_precision', cv=cv, n_jobs=1, refit=True, verbose=0, return_train_score=False)
    search.fit(X_train, y_train)
    searches[name] = search
    best_params[name] = {key.replace('model__',''): value for key,value in search.best_params_.items()}
    validation_probabilities[name] = predict_prob(search.best_estimator_, X_val)
    print(name, '| best CV AUPRC:', round(float(search.best_score_),4), '| params:', best_params[name])

# Historical-only sanity check (plan addendum §2): no later-cycle data are touched here.
print('\nHistorical validation metrics (sanity check, never a test-set gate):')
val_metrics = pd.DataFrame([{'model':name, **metric_pack(y_val, p)} for name,p in validation_probabilities.items()])
display(val_metrics)
val_metrics.to_csv(ARTIFACT_DIR/'historical_validation_metrics.csv', index=False)

val_onto = hist_onto.iloc[val_idx]
val_rule_score = val_onto['onto_rule_priority_score'].to_numpy(dtype=float)
alpha_candidates = [0.0,0.025,0.05,0.10,0.15,0.20,0.30]
alpha_results=[]
for alpha in alpha_candidates:
    result=capacity_pack(y_val, validation_probabilities['xgboost_raw'] + alpha*val_rule_score, PRIMARY_CAPACITY)
    alpha_results.append({'alpha':alpha,**result})
alpha_results=sorted(alpha_results,key=lambda r:(r['sensitivity'] if np.isfinite(r['sensitivity']) else -1,r['precision'] if np.isfinite(r['precision']) else -1,-r['alpha']),reverse=True)
HYBRID_ALPHA=float(alpha_results[0]['alpha'])
print('Historical validation selects hybrid alpha:', HYBRID_ALPHA)
display(pd.DataFrame(alpha_results))

# Prespecified race_ethnicity_ablation: remove only the higher-risk ethnicity risk-factor flag.
# The ADA BMI threshold adaptation for Asian ancestry remains in this specific ablation and is disclosed.
val_onto_no_ethnicity = hist_onto_no_ethnicity.iloc[val_idx]
val_rule_score_no_ethnicity = val_onto_no_ethnicity['onto_rule_priority_score'].to_numpy(dtype=float)
alpha_results_no_ethnicity=[]
for alpha in alpha_candidates:
    result=capacity_pack(y_val, validation_probabilities['xgboost_raw'] + alpha*val_rule_score_no_ethnicity, PRIMARY_CAPACITY)
    alpha_results_no_ethnicity.append({'alpha':alpha,**result})
alpha_results_no_ethnicity=sorted(alpha_results_no_ethnicity,key=lambda r:(r['sensitivity'] if np.isfinite(r['sensitivity']) else -1,r['precision'] if np.isfinite(r['precision']) else -1,-r['alpha']),reverse=True)
HYBRID_ALPHA_NO_ETHNICITY=float(alpha_results_no_ethnicity[0]['alpha'])
print('Historical validation selects no-higher-risk-ethnicity alpha:', HYBRID_ALPHA_NO_ETHNICITY)

# Smallest detectable difference on historical validation (plan addendum §7).
MDE = {}
MDE['primary_hybrid_vs_xgb_raw'] = paired_capacity_bootstrap(y_val, validation_probabilities['xgboost_raw'] + HYBRID_ALPHA*val_rule_score, validation_probabilities['xgboost_raw'])
MDE['logistic_ontology_vs_raw'] = paired_capacity_bootstrap(y_val, validation_probabilities['logistic_ontology'], validation_probabilities['logistic_raw'])
MDE['xgboost_ontology_vs_raw'] = paired_capacity_bootstrap(y_val, validation_probabilities['xgboost_ontology'], validation_probabilities['xgboost_raw'])
for key, res in MDE.items():
    res['mde_95'] = float(1.96 * res['std_error']) if np.isfinite(res.get('std_error', np.nan)) else float('nan')
    print(f"Validation MDE for {key}: SE={res['std_error']:.4f}, MDE95={res['mde_95']:.4f}, point={res['difference']:.4f}")
print('Effects smaller than the MDE are reported as below the resolution of this design.')

In [ ]:
# Final training: refit selected configurations on all historical data across five seeds.
# The 2021–2023 temporal test is not used for selecting hyperparameters or alpha.
# LR and XGBoost defaults are deterministic; the seeds only affect XGBoost subsampling.
models = {name: [] for name in X_variants}
test_raw = temporal_test[RAW_FEATURES].copy()
test_onto = derive_ontology_features(test_raw)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
test_onto_no_ethnicity = derive_ontology_features(test_raw, include_ethnicity=False)[RAW_FEATURES + ONTOLOGY_FEATURES].copy()
test_manual = derive_manual_control(test_raw)[RAW_FEATURES + MANUAL_FEATURES].copy()
y_test = temporal_test['target'].astype(int).to_numpy()
X_hist_variants = {
    'logistic_raw': (hist_raw, RAW_FEATURES, 'logistic'),
    'xgboost_raw': (hist_raw, RAW_FEATURES, 'xgboost'),
    'logistic_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'logistic'),
    'xgboost_ontology': (hist_onto, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_ontology_no_ethnicity': (hist_onto_no_ethnicity, RAW_FEATURES+ONTOLOGY_FEATURES, 'xgboost'),
    'xgboost_generic_control': (hist_manual, RAW_FEATURES+MANUAL_FEATURES, 'xgboost'),
}
for seed in SEEDS:
    print('Training seed', seed)
    for name, (X_train_full, features, family) in X_hist_variants.items():
        params = best_params[name]
        pipeline = make_pipeline(family, features, seed=seed, params=params)
        pipeline.fit(X_train_full[features], y_all)
        models[name].append(pipeline)

X_test_variants = {
    'logistic_raw': test_raw[RAW_FEATURES],
    'xgboost_raw': test_raw[RAW_FEATURES],
    'logistic_ontology': test_onto[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_ontology': test_onto[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_ontology_no_ethnicity': test_onto_no_ethnicity[RAW_FEATURES+ONTOLOGY_FEATURES],
    'xgboost_generic_control': test_manual[RAW_FEATURES+MANUAL_FEATURES],
}
probabilities = {name: np.mean(np.vstack([predict_prob(m, X_test_variants[name]) for m in model_list]),axis=0) for name, model_list in models.items()}
metrics_by_model = {}
per_seed_metrics = {}
for name, model_list in models.items():
    seed_metrics=[metric_pack(y_test,predict_prob(model,X_test_variants[name])) for model in model_list]
    per_seed_metrics[name]=seed_metrics
    metrics_by_model[name]={}
    for metric in ['auroc','auprc','brier']:
        vals=np.asarray([m[metric] for m in seed_metrics],dtype=float)
        metrics_by_model[name][metric+'_mean']=float(np.nanmean(vals))
        metrics_by_model[name][metric+'_sd']=float(np.nanstd(vals,ddof=1))
    metrics_by_model[name]['ensemble'] = metric_pack(y_test, probabilities[name])
    metrics_by_model[name]['cv_best_params'] = best_params[name]

display(pd.DataFrame([{ 'model':k, **{m:v for m,v in val.items() if m.endswith('_mean') or m.endswith('_sd')} } for k,val in metrics_by_model.items()]))
print('Temporal test participants:',len(y_test),'| positive cases:',int(y_test.sum()),'| prevalence:',round(float(y_test.mean()),4))
print('Ontology rule coverage on the temporal test cohort:',
      {'eligible': float(test_onto['onto_screening_eligible'].mean(skipna=True)),
       'not_assessable_fraction': float(test_onto['onto_screening_eligible'].isna().mean()),
       'criteria_missing_mean': float(test_onto['onto_criteria_missing_count'].mean())})

In [ ]:
# Locked temporal-test evaluation: fixed-capacity policies, uncertainty, and exploratory analyses.
# This is the only stage that reads 2021–2023 outcomes for evaluation.
test_rule_score = test_onto['onto_rule_priority_score'].to_numpy(dtype=float)
test_rule_score_no_ethnicity = test_onto_no_ethnicity['onto_rule_priority_score'].to_numpy(dtype=float)
test_eligible = test_onto['onto_screening_eligible'].to_numpy(dtype=float)
hybrid_score = probabilities['xgboost_raw'] + HYBRID_ALPHA*test_rule_score
hybrid_score_no_ethnicity = probabilities['xgboost_raw'] + HYBRID_ALPHA_NO_ETHNICITY*test_rule_score_no_ethnicity
published_score = cdc_ada_risk_score(test_raw).to_numpy(dtype=float)

screening_budget_rows=[]
policy_scores={
    'XGBoost raw': probabilities['xgboost_raw'],
    'XGBoost + ontology features': probabilities['xgboost_ontology'],
    'XGBoost + ontology features (no higher-risk ethnicity flag)': probabilities['xgboost_ontology_no_ethnicity'],
    'XGBoost + ontology policy': hybrid_score,
    'XGBoost + ontology policy (no higher-risk ethnicity flag)': hybrid_score_no_ethnicity,
    'Logistic raw': probabilities['logistic_raw'],
    'Logistic + ontology features': probabilities['logistic_ontology'],
    'Ontology-only rule score': test_rule_score,
    'Ontology-only rule score (no higher-risk ethnicity flag)': test_rule_score_no_ethnicity,
    'XGBoost + generic equivalent features': probabilities['xgboost_generic_control'],
    'CDC/ADA prediabetes risk test (published score)': published_score,
}
for capacity in CAPACITIES:
    for policy,score in policy_scores.items():
        screening_budget_rows.append({'policy':policy,**capacity_pack(y_test,score,capacity)})
screening_budget_df=pd.DataFrame(screening_budget_rows)
display(screening_budget_df)

# H2, confirmatory: hybrid minus XGBoost-only sensitivity at 20% capacity.
primary_effect=paired_capacity_bootstrap(y_test,probabilities['xgboost_raw'],hybrid_score)
base_at20=capacity_pack(y_test,probabilities['xgboost_raw'],PRIMARY_CAPACITY)
hybrid_at20=capacity_pack(y_test,hybrid_score,PRIMARY_CAPACITY)
primary_effect.update({'baseline_sensitivity':base_at20['sensitivity'],'hybrid_sensitivity':hybrid_at20['sensitivity'],
                       'baseline_positive_found':base_at20['positive_found'],'hybrid_positive_found':hybrid_at20['positive_found'],
                       'baseline_precision':base_at20['precision'],'hybrid_precision':hybrid_at20['precision'],
                       'prevalence':base_at20['prevalence'],'sensitivity_ceiling':base_at20['sensitivity_ceiling'],
                       'fraction_of_ceiling_baseline':base_at20['fraction_of_ceiling'],
                       'fraction_of_ceiling_hybrid':hybrid_at20['fraction_of_ceiling'],
                       'mde_95_from_historical_validation':MDE['primary_hybrid_vs_xgb_raw']['mde_95'],
                       'capacity':PRIMARY_CAPACITY})
print('H2 primary endpoint (confirmatory), hybrid minus ML-only sensitivity:',primary_effect)

no_eth_base_at20=capacity_pack(y_test,probabilities['xgboost_raw'],PRIMARY_CAPACITY)
no_eth_hybrid_at20=capacity_pack(y_test,hybrid_score_no_ethnicity,PRIMARY_CAPACITY)
primary_effect_no_ethnicity=paired_capacity_bootstrap(y_test,probabilities['xgboost_raw'],hybrid_score_no_ethnicity)
primary_effect_no_ethnicity.update({'baseline_sensitivity':no_eth_base_at20['sensitivity'],'hybrid_sensitivity':no_eth_hybrid_at20['sensitivity'],
                                    'baseline_positive_found':no_eth_base_at20['positive_found'],'hybrid_positive_found':no_eth_hybrid_at20['positive_found'],
                                    'capacity':PRIMARY_CAPACITY})
print('Primary endpoint after removing the higher-risk ethnicity rule flag:',primary_effect_no_ethnicity)

# E3, exploratory: does the knowledge layer change ranking metrics, and does it help one family more?
paired_effects = {
    'logistic_ontology_vs_raw_auprc': paired_metric_bootstrap(y_test, probabilities['logistic_ontology'], probabilities['logistic_raw'], 'auprc'),
    'logistic_ontology_vs_raw_auroc': paired_metric_bootstrap(y_test, probabilities['logistic_ontology'], probabilities['logistic_raw'], 'auroc'),
    'xgboost_ontology_vs_raw_auprc': paired_metric_bootstrap(y_test, probabilities['xgboost_ontology'], probabilities['xgboost_raw'], 'auprc'),
    'xgboost_ontology_vs_raw_auroc': paired_metric_bootstrap(y_test, probabilities['xgboost_ontology'], probabilities['xgboost_raw'], 'auroc'),
    'logistic_ontology_vs_raw_sens20': paired_capacity_bootstrap(y_test, probabilities['logistic_ontology'], probabilities['logistic_raw']),
    'xgboost_ontology_vs_raw_sens20': paired_capacity_bootstrap(y_test, probabilities['xgboost_ontology'], probabilities['xgboost_raw']),
    'xgboost_ontology_no_ethnicity_vs_raw_sens20': paired_capacity_bootstrap(y_test, probabilities['xgboost_ontology_no_ethnicity'], probabilities['xgboost_raw']),
}
MDE_KEY_FOR = {
    'logistic_ontology_vs_raw_auprc': 'logistic_ontology_vs_raw',
    'logistic_ontology_vs_raw_auroc': 'logistic_ontology_vs_raw',
    'logistic_ontology_vs_raw_sens20': 'logistic_ontology_vs_raw',
    'xgboost_ontology_vs_raw_auprc': 'xgboost_ontology_vs_raw',
    'xgboost_ontology_vs_raw_auroc': 'xgboost_ontology_vs_raw',
    'xgboost_ontology_vs_raw_sens20': 'xgboost_ontology_vs_raw',
    'xgboost_ontology_no_ethnicity_vs_raw_sens20': 'xgboost_ontology_vs_raw',
}
for key, res in paired_effects.items():
    res['mde_95_from_historical_validation'] = MDE[MDE_KEY_FOR[key]]['mde_95']
    print(key, {k: round(v,4) if isinstance(v,float) else v for k,v in res.items() if k in ('difference','ci_low','ci_high','std_error','mde_95_from_historical_validation')})

# H3, exploratory: guideline disagreement vs a size-matched confidence-only review set.
ml_topk = np.argsort(-probabilities['xgboost_raw'], kind='mergesort')[:base_at20['n_selected']]
rule_topk = np.argsort(-np.nan_to_num(test_rule_score, nan=-np.inf), kind='mergesort')[:base_at20['n_selected']]
in_ml = np.zeros(len(y_test), dtype=bool); in_ml[ml_topk] = True
in_rule = np.zeros(len(y_test), dtype=bool); in_rule[rule_topk] = True
disagreement = in_ml ^ in_rule
cutoff = float(np.sort(probabilities['xgboost_raw'])[::-1][base_at20['n_selected']-1])
confidence_rank = np.argsort(np.abs(probabilities['xgboost_raw'] - cutoff), kind='mergesort')[:int(disagreement.sum())]
review_summary = {
    'capacity': PRIMARY_CAPACITY,
    'ml_rule_overlap': int((in_ml & in_rule).sum()),
    'disagreement_set_size': int(disagreement.sum()),
    'disagreement_positives': int(y_test[disagreement].sum()),
    'disagreement_precision': float(y_test[disagreement].mean()) if disagreement.any() else None,
    'ml_set_positives': int(y_test[in_ml].sum()),
    'rule_set_positives': int(y_test[in_rule].sum()),
    'confidence_review_size': int(len(confidence_rank)),
    'confidence_review_positives': int(y_test[confidence_rank].sum()),
    'confidence_review_precision': float(y_test[confidence_rank].mean()) if len(confidence_rank) else None,
    'prevalence': float(y_test.mean()),
}
print('Disagreement vs size-matched confidence-only review:', review_summary)

# Exploratory subgroup summary at 20% capacity for the two headline policies.
def subgroup_rows(df, scores_dict):
    rows=[]
    age = pd.to_numeric(df['age_years'], errors='coerce')
    sex = pd.to_numeric(df['sex_code'], errors='coerce')
    race = pd.to_numeric(df['race_ethnicity_code'], errors='coerce')
    groups = {
        'all': np.ones(len(df), dtype=bool),
        'age_18_39': (age < 40), 'age_40_59': ((age >= 40) & (age < 60)), 'age_60_plus': (age >= 60),
        'female': (sex == 2), 'male': (sex == 1),
        'hispanic': race.isin([1, 2]), 'nh_black': (race == 4), 'nh_white': (race == 3), 'nh_asian': (race == 6),
    }
    for gname, mask in groups.items():
        mask = mask.fillna(False).to_numpy()
        if mask.sum() < 50 or y_test[mask].sum() < 10:
            continue
        for pname, score in scores_dict.items():
            cap = capacity_pack(y_test[mask], score[mask], PRIMARY_CAPACITY)
            rows.append({'subgroup': gname, 'policy': pname, 'n': int(mask.sum()), 'positive_total': cap['positive_total'],
                         'prevalence': cap['prevalence'], 'sensitivity': cap['sensitivity'], 'precision': cap['precision']})
    return pd.DataFrame(rows)

subgroup_df = subgroup_rows(temporal_test, {'XGBoost raw': probabilities['xgboost_raw'], 'XGBoost + ontology features': probabilities['xgboost_ontology']})
print('\nSubgroup summary at 20% capacity (exploratory; only groups with >=50 participants and >=10 positives):')
display(subgroup_df)

In [ ]:
# E1 (plan addendum §3): low-data learning curve. Exploratory; reported whatever direction it takes.
e1_rows = []
for frac in LOW_DATA_FRACTIONS:
    if frac >= 1.0:
        splits = [(np.arange(len(y_all)), None)]
    else:
        sss = StratifiedShuffleSplit(n_splits=LOW_DATA_REPEATS, train_size=frac, random_state=42)
        splits = list(sss.split(hist_raw, y_all))
    for rep_idx, (sub_idx, _) in enumerate(splits):
        y_sub = y_all[sub_idx]
        for family, features_df, feature_sets in [
            ('logistic', hist_raw, {'raw': RAW_FEATURES, 'ontology': RAW_FEATURES + ONTOLOGY_FEATURES}),
            ('xgboost', hist_raw, {'raw': RAW_FEATURES, 'ontology': RAW_FEATURES + ONTOLOGY_FEATURES}),
        ]:
            for set_name, cols in feature_sets.items():
                if set_name == 'raw':
                    X_sub = hist_raw.iloc[sub_idx][cols]; X_full_test = test_raw[cols]
                else:
                    X_sub = hist_onto.iloc[sub_idx][cols]; X_full_test = test_onto[cols]
                pipe = make_pipeline(family, cols, seed=42, params=best_params['xgboost_raw' if family=='xgboost' else 'logistic_raw'])
                pipe.fit(X_sub, y_sub)
                p = predict_prob(pipe, X_full_test)
                cls = metric_pack(y_test, p); cap = capacity_pack(y_test, p, PRIMARY_CAPACITY)
                e1_rows.append({'fraction': frac, 'repeat': rep_idx, 'model': family, 'feature_set': set_name,
                                'n_train': int(len(sub_idx)), 'test_auroc': cls['auroc'], 'test_auprc': cls['auprc'],
                                'test_brier': cls['brier'], 'cap_sensitivity': cap['sensitivity'],
                                'cap_precision': cap['precision']})
e1_df = pd.DataFrame(e1_rows)
e1_summary = e1_df.groupby(['fraction','model','feature_set']).agg(
    auprc_mean=('test_auprc','mean'), auprc_sd=('test_auprc','std'),
    auroc_mean=('test_auroc','mean'), auroc_sd=('test_auroc','std'),
    sens_mean=('cap_sensitivity','mean'), sens_sd=('cap_sensitivity','std'),
    n_train=('n_train','mean')).reset_index()
e1_wide = e1_summary.pivot_table(index=['fraction','model','n_train'], columns='feature_set',
                                 values=['auprc_mean','sens_mean'])
print('E1 low-data learning curve (mean over subsamples, locked temporal test):')
display(e1_wide.round(4))
e1_delta = e1_summary.pivot_table(index=['fraction','model'], columns='feature_set', values='auprc_mean')
e1_delta['auprc_delta_ontology_minus_raw'] = e1_delta['ontology'] - e1_delta['raw']
e1_sens_delta = e1_summary.pivot_table(index=['fraction','model'], columns='feature_set', values='sens_mean')
e1_delta['sens20_delta_ontology_minus_raw'] = e1_sens_delta['ontology'] - e1_sens_delta['raw']
print('\nE1 gain from guideline features (positive = knowledge helped):')
display(e1_delta.round(4))
e1_df.to_csv(ARTIFACT_DIR/'e1_low_data_results.csv', index=False)
e1_summary.to_csv(ARTIFACT_DIR/'e1_low_data_summary.csv', index=False)

In [ ]:
# E2 (plan addendum §3): missing-input stress test. Exploratory.
# Masked inputs are re-derived into ontology features, so the rule layer never sees information
# the ML system does not receive; the executor reports criteria that are not assessable instead of guessing.
e2_rows = []
not_assessable_by_rate = {}
for rate in MASK_RATES:
    for rep in range(MASK_REPEATS):
        rng = np.random.default_rng(42 + rep)
        masked_raw = test_raw.copy()
        if rate > 0:
            for col in MASK_COLUMNS:
                if col in masked_raw.columns:
                    drop = rng.random(len(masked_raw)) < rate
                    masked_raw.loc[drop, col] = np.nan
        masked_onto = derive_ontology_features(masked_raw)[RAW_FEATURES + ONTOLOGY_FEATURES]
        not_assessable = float(masked_onto['onto_screening_eligible'].isna().mean())
        not_assessable_by_rate.setdefault(rate, []).append(not_assessable)
        variants = {
            'logistic_raw': (masked_raw[RAW_FEATURES], 'logistic'),
            'logistic_ontology': (masked_onto[RAW_FEATURES + ONTOLOGY_FEATURES], 'logistic'),
            'xgboost_raw': (masked_raw[RAW_FEATURES], 'xgboost'),
            'xgboost_ontology': (masked_onto[RAW_FEATURES + ONTOLOGY_FEATURES], 'xgboost'),
        }
        probs = {}
        for name, (X_m, family) in variants.items():
            probs[name] = np.mean(np.vstack([predict_prob(m, X_m) for m in models[name]]), axis=0)
        probs['xgboost_ontology_policy'] = probs['xgboost_raw'] + HYBRID_ALPHA * masked_onto['onto_rule_priority_score'].to_numpy(dtype=float)
        for name, p in probs.items():
            cls = metric_pack(y_test, p); cap = capacity_pack(y_test, p, PRIMARY_CAPACITY)
            e2_rows.append({'mask_rate': rate, 'repeat': rep, 'model': name, 'not_assessable_fraction': not_assessable,
                            'test_auroc': cls['auroc'], 'test_auprc': cls['auprc'], 'test_brier': cls['brier'],
                            'cap_sensitivity': cap['sensitivity'], 'cap_precision': cap['precision']})
e2_df = pd.DataFrame(e2_rows)
e2_summary = e2_df.groupby(['mask_rate','model']).agg(
    auprc_mean=('test_auprc','mean'), auprc_sd=('test_auprc','std'),
    auroc_mean=('test_auroc','mean'), brier_mean=('test_brier','mean'),
    sens_mean=('cap_sensitivity','mean'), sens_sd=('cap_sensitivity','std'),
    not_assessable_mean=('not_assessable_fraction','mean')).reset_index()
print('E2 missing-input stress test (mean over masked repeats):')
display(e2_summary.round(4))
e2_pivot = e2_summary.pivot_table(index=['mask_rate','model'], values=['auprc_mean','sens_mean'])
print('\nE2 degradation relative to the unmasked run:')
base = e2_summary[e2_summary['mask_rate']==0.0].set_index('model')
degraded = e2_summary.copy()
degraded['auprc_change_vs_unmasked'] = degraded['auprc_mean'] - degraded['model'].map(base['auprc_mean'])
degraded['sens_change_vs_unmasked'] = degraded['sens_mean'] - degraded['model'].map(base['sens_mean'])
display(degraded[['mask_rate','model','auprc_mean','auprc_change_vs_unmasked','sens_mean','sens_change_vs_unmasked','not_assessable_mean']].round(4))
e2_df.to_csv(ARTIFACT_DIR/'e2_missing_input_results.csv', index=False)
e2_summary.to_csv(ARTIFACT_DIR/'e2_missing_input_summary.csv', index=False)
json.dump({str(k): v for k, v in not_assessable_by_rate.items()}, open(ARTIFACT_DIR/'e2_not_assessable_rates.json','w'), indent=2)

In [ ]:
# Published-knowledge baseline (plan addendum §5): the official CDC/ADA prediabetes risk test.
# Scoring source: CDC/ADA Prediabetes Risk Test (adapted from Bang et al., Ann Intern Med 2009).
# Items unavailable in the harmonized NHANES files are scored 0 and listed, never silently replaced.
val_published_score = cdc_ada_risk_score(historical.iloc[val_idx]).to_numpy(dtype=float)
published_rows = [
    {'cohort': 'historical_validation', **metric_pack(y_val, val_published_score)},
    {'cohort': 'temporal_test', **metric_pack(y_test, published_score)},
]
for capacity in CAPACITIES:
    published_rows.append({'cohort': 'temporal_test', 'capacity': capacity, **capacity_pack(y_test, published_score, capacity)})
published_df = pd.DataFrame(published_rows)
print('CDC/ADA published risk test, mapped to NHANES (unavailable items scored 0):')
display(published_df.round(4))
print('Unavailable items in the mapping:', CDC_ADA_UNAVAILABLE_ITEMS)
print('High-risk threshold on the published scale is a total score >= 5; the ranking analysis uses the raw point total.')
high_risk = (published_score >= 5).astype(int)
print('Temporal test at the published >=5 threshold:',
      {'n_flagged': int(high_risk.sum()), 'positives_captured': int(y_test[high_risk].sum()),
       'precision': float(y_test[high_risk].mean()) if high_risk.any() else None,
       'sensitivity': float(y_test[high_risk].sum() / max(1, y_test.sum()))})
published_df.to_csv(ARTIFACT_DIR/'published_score_baseline.csv', index=False)

In [ ]:
# Research figures from actual temporal-test predictions.
figure_paths=[]
# ROC and PR curves
fig, axes = plt.subplots(1,2,figsize=(12,5))
plot_models = ['logistic_raw','xgboost_raw','logistic_ontology','xgboost_ontology','xgboost_ontology_no_ethnicity','xgboost_generic_control']
labels = {'logistic_raw':'Logistic raw','logistic_ontology':'Logistic + ontology features','xgboost_raw':'XGBoost raw',
          'xgboost_ontology':'XGBoost + ontology features','xgboost_ontology_no_ethnicity':'XGBoost + ontology (no ethnicity flag)',
          'xgboost_generic_control':'XGBoost + generic control'}
for name in plot_models:
    p=probabilities[name]
    fpr,tpr,_=roc_curve(y_test,p)
    prec,rec,_=precision_recall_curve(y_test,p)
    axes[0].plot(fpr,tpr,label=f"{labels[name]} (AUC {roc_auc_score(y_test,p):.3f})")
    axes[1].plot(rec,prec,label=f"{labels[name]} (AP {average_precision_score(y_test,p):.3f})")
axes[0].plot([0,1],[0,1],'--',color='gray',linewidth=1); axes[0].set_title('ROC curve — NHANES 2021–2023 temporal test'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[1].set_title('Precision–recall curve — NHANES 2021–2023 temporal test'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'roc_pr_curves.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Calibration plot
fig,ax=plt.subplots(figsize=(6,5))
for name in plot_models:
    frac,mean_pred=calibration_curve(y_test,probabilities[name],n_bins=10,strategy='quantile')
    ax.plot(mean_pred,frac,marker='o',label=labels[name])
ax.plot([0,1],[0,1],'--',color='gray'); ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed positive fraction'); ax.set_title('Calibration on temporal test'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'calibration.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Fixed screening capacity sensitivity
fig,ax=plt.subplots(figsize=(9,5.5))
for policy,group in screening_budget_df.groupby('policy'):
    group=group.sort_values('capacity')
    ax.plot(group['capacity']*100,group['sensitivity'],marker='o',label=policy)
ax.plot([c*100 for c in CAPACITIES], [min(1.0, c/float(y_test.mean())) for c in CAPACITIES], 'k--', label='Capacity ceiling')
ax.set_xlabel('Screening capacity (% of eligible test cohort)'); ax.set_ylabel('Sensitivity / fraction of positive cases found'); ax.set_ylim(0,1); ax.set_title('Sensitivity at fixed screening capacity'); ax.legend(fontsize=7); ax.grid(alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'capacity_sensitivity.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Primary effect interval
fig,ax=plt.subplots(figsize=(7,2.8))
ax.errorbar([primary_effect['difference']],[0],xerr=[[primary_effect['difference']-primary_effect['ci_low']],[primary_effect['ci_high']-primary_effect['difference']]],fmt='o',capsize=6)
ax.axvline(0,color='gray',linestyle='--'); ax.set_yticks([]); ax.set_xlabel('Hybrid minus ML-only sensitivity at 20% capacity'); ax.set_title('Paired bootstrap 95% interval (H2)'); ax.grid(axis='x',alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'primary_effect_ci.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# E1 learning curve
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax, metric, title in zip(axes, ['auprc_mean','sens_mean'], ['AUPRC on temporal test','Sensitivity at 20% capacity']):
    for family, style in [('logistic','-o'), ('xgboost','-s')]:
        for set_name, color in [('raw','#64748b'), ('ontology','#2563eb')]:
            sub = e1_summary[(e1_summary['model']==family) & (e1_summary['feature_set']==set_name)].sort_values('fraction')
            ax.plot(sub['fraction']*100, sub[metric], style, color=color, label=f'{family} {set_name}')
    ax.set_xlabel('Fraction of historical training data (%)'); ax.set_title(title); ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); pth=ARTIFACT_DIR/'e1_learning_curve.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# E2 missing-input stress
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for name, label, style in [('logistic_raw','Logistic raw','--o'), ('logistic_ontology','Logistic + ontology','-o'),
                           ('xgboost_raw','XGBoost raw','--s'), ('xgboost_ontology','XGBoost + ontology','-s')]:
    sub = e2_summary[e2_summary['model']==name].sort_values('mask_rate')
    axes[0].plot(sub['mask_rate']*100, sub['auprc_mean'], style, label=label)
    axes[1].plot(sub['mask_rate']*100, sub['sens_mean'], style, label=label)
axes[0].set_xlabel('MCAR masking rate of selected inputs (%)'); axes[0].set_ylabel('AUPRC'); axes[0].set_title('E2 discrimination under missing inputs')
axes[1].set_xlabel('MCAR masking rate of selected inputs (%)'); axes[1].set_ylabel('Sensitivity at 20% capacity'); axes[1].set_title('E2 fixed-capacity screening under missing inputs')
for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); pth=ARTIFACT_DIR/'e2_missing_input.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

# Model-family deltas (E3)
fig,ax=plt.subplots(figsize=(7,4))
names = list(paired_effects.keys())
point = [paired_effects[k]['difference'] for k in names]
low = [paired_effects[k]['ci_low'] for k in names]
high = [paired_effects[k]['ci_high'] for k in names]
ypos = np.arange(len(names))
ax.errorbar(point, ypos, xerr=[np.array(point)-np.array(low), np.array(high)-np.array(point)], fmt='o', capsize=4)
ax.axvline(0, color='gray', linestyle='--')
ax.set_yticks(ypos); ax.set_yticklabels([n.replace('_vs_raw','\nvs raw').replace('_ontology',' + ontology').replace('sens20','sens@20%') for n in names], fontsize=8)
ax.set_xlabel('Difference on the temporal test (95% paired bootstrap interval)'); ax.set_title('E3: effect of guideline features by model family')
ax.grid(axis='x', alpha=.25)
plt.tight_layout(); pth=ARTIFACT_DIR/'e3_model_family_deltas.png'; fig.savefig(pth,dpi=180,bbox_inches='tight'); figure_paths.append(str(pth)); plt.show()

In [ ]:
# Persist research artifacts and provenance.
import pandas as pd
research_results = {
    'protocol': {**PROTOCOL, 'python': platform.python_version(), 'pandas': pd.__version__,
                 'sklearn': __import__('sklearn').__version__, 'xgboost': __import__('xgboost').__version__,
                 'rdflib': __import__('rdflib').__version__, 'seeds': SEEDS,
                 'primary_capacity': PRIMARY_CAPACITY, 'n_bootstrap': N_BOOTSTRAP,
                 'hybrid_alpha': HYBRID_ALPHA, 'hybrid_alpha_no_ethnicity': HYBRID_ALPHA_NO_ETHNICITY,
                 'hybrid_alpha_candidates': alpha_results,
                 'cdc_ada_unavailable_items': CDC_ADA_UNAVAILABLE_ITEMS,
                 'rule_sections': RULE_SECTIONS},
    'data': {'cohort_flow': COHORT_FLOW, 'qc': cohort_qc, 'composition': composition_df.to_dict('records')},
    'models': metrics_by_model, 'best_params': best_params,
    'capacity_metrics': {policy: [capacity_pack(y_test, score, c) for c in CAPACITIES] for policy, score in policy_scores.items()},
    'primary_effect': primary_effect, 'primary_effect_no_ethnicity': primary_effect_no_ethnicity,
    'paired_effects': paired_effects, 'mde': MDE,
    'review_summary': review_summary, 'subgroups': subgroup_df.to_dict('records'),
    'e1': e1_summary.to_dict('records'), 'e2': e2_summary.to_dict('records'),
    'published_baseline': published_df.to_dict('records'),
    'test_n': int(len(y_test)), 'test_positive_count': int(y_test.sum()), 'test_prevalence': float(y_test.mean()),
    'historical_n': int(len(historical)), 'historical_prevalence': float(historical['target'].mean()),
    'feature_names': {'raw': RAW_FEATURES, 'ontology': ONTOLOGY_FEATURES, 'manual': MANUAL_FEATURES},
}
(ARTIFACT_DIR / 'research_results.json').write_text(json.dumps(research_results, indent=2, default=str), encoding='utf-8')

feature_manifest = {
    'raw_numeric': RAW_NUMERIC, 'raw_categorical': RAW_CATEGORICAL, 'raw_features': RAW_FEATURES,
    'ontology_features': ONTOLOGY_FEATURES, 'manual_features': MANUAL_FEATURES,
    'forbidden_tokens': list(FORBIDDEN_FEATURE_TOKENS),
    'unharmonized_variables': unharmonized,
}
(ARTIFACT_DIR / 'feature_manifest.json').write_text(json.dumps(feature_manifest, indent=2), encoding='utf-8')

test_predictions = temporal_test[['SEQN']].copy()
test_predictions['target'] = y_test
test_predictions['hba1c_percent'] = pd.to_numeric(temporal_test['hba1c_percent'], errors='coerce')
for name, p in probabilities.items():
    test_predictions[name] = p
test_predictions['hybrid_score'] = hybrid_score
test_predictions['hybrid_score_no_ethnicity'] = hybrid_score_no_ethnicity
test_predictions['cdc_ada_score'] = published_score
test_predictions['onto_rule_priority_score'] = test_rule_score
test_predictions['disagreement_review'] = disagreement
test_predictions.to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)

bundle = {'models': models, 'best_params': best_params, 'hybrid_alpha': HYBRID_ALPHA,
          'hybrid_alpha_no_ethnicity': HYBRID_ALPHA_NO_ETHNICITY, 'raw_features': RAW_FEATURES,
          'ontology_features': ONTOLOGY_FEATURES, 'manual_features': MANUAL_FEATURES,
          'ontology_ttl': ontology_path.read_text(encoding='utf-8') if ontology_path else ONTOLOGY_TTL_EMBEDDED,
          'cdc_ada_unavailable_items': CDC_ADA_UNAVAILABLE_ITEMS}
joblib.dump(bundle, ARTIFACT_DIR / 'model_bundle.joblib')

run_manifest = {
    'protocol': PROTOCOL, 'artifact_dir': str(ARTIFACT_DIR),
    'figure_paths': figure_paths,
    'csv_paths': [str(p) for p in sorted(ARTIFACT_DIR.glob('*.csv'))],
    'json_paths': [str(p) for p in sorted(ARTIFACT_DIR.glob('*.json'))],
    'generated_utc': datetime.now(timezone.utc).isoformat(),
}
(ARTIFACT_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2, default=str), encoding='utf-8')

zip_path = ARTIFACT_DIR / 'clinontoscreen_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ARTIFACT_DIR.rglob('*')):
        if path.is_file() and path != zip_path:
            zf.write(path, path.relative_to(ARTIFACT_DIR))
print('Artifacts written to', ARTIFACT_DIR)
for path in sorted(ARTIFACT_DIR.rglob('*')):
    if path.is_file():
        print(' ', path.relative_to(ARTIFACT_DIR), f'{path.stat().st_size/1024:.1f} KB')

## Using the outputs

- `research_results.json` is the machine-readable record of every reported number, including
  the protocol commit/tag, hybrid alpha selection, paired bootstrap intervals, smallest
  detectable differences, E1/E2/E3 summaries, and the published CDC/ADA baseline.
- `test_predictions.csv` holds one row per 2021–2023 participant with the locked label, every
  model probability, the hybrid scores, the CDC/ADA point total, the ontology rule score, and
  the disagreement-review flag.
- `model_bundle.joblib` reloads the five-seed ensembles, best hyperparameters, hybrid alphas,
  and the ontology text used at training time:

```python
import joblib, pandas as pd
bundle = joblib.load('artifacts/model_bundle.joblib')
proba = bundle['models']['xgboost_ontology'][0].predict_proba(X[bundle['raw_features'] + bundle['ontology_features']])[:, 1]
```

- `capacity_metrics` in `research_results.json` is the fixed-budget view: for each policy and
  capacity it reports sensitivity, its prevalence ceiling, the fraction of that ceiling
  achieved, PPV, and lift over random referral.
- The ontology shipped with the run is `diabetes_screening.ttl`; the `sectionReference` field
  on each rule and outcome definition records which guideline section justifies it, and
  `operationalizationStatus` records which criteria are described but not computable from
  NHANES (family history, gestational diabetes, PCOS).